# Pest detection: validated and reproducible workflow

This notebook prepares an explicitly supplied COCO dataset and optionally trains YOLO. It does not download data, install packages, or fabricate metrics. Missing inputs and artifacts stop with a clear error.

**Important:** validation metrics are not a final unbiased test score; this repository has no separate held-out test set.

In [ ]:
import os
from pathlib import Path

from pest_detection_pipeline import (
    DataUnavailableError,
    convert_coco_to_yolo,
    find_corrupt_images,
    load_coco,
    split_dataset,
    training_seed,
    validate_dataset_inputs,
    write_dataset_yaml,
)

SEED = 42
dataset_root_value = os.environ.get("PEST_DATASET_ROOT")
if not dataset_root_value:
    raise DataUnavailableError(
        "PEST_DATASET_ROOT is not set. Provide the external dataset; no data is generated."
    )
DATASET_ROOT = Path(dataset_root_value).expanduser()
SOURCE_ROOT = DATASET_ROOT / "ImageClassesCombinedWithCOCOAnnotations"
COCO_JSON = SOURCE_ROOT / "coco_instances.json"
IMAGE_ROOT = SOURCE_ROOT / "images_raw"
OUTPUT_DIR = Path(os.environ.get("PEST_OUTPUT_DIR", "yolo_dataset")).expanduser()
RUN_TRAINING = os.environ.get("PEST_RUN_TRAINING", "0") == "1"


In [ ]:
summary = validate_dataset_inputs(COCO_JSON, IMAGE_ROOT)
corrupt_images = find_corrupt_images(IMAGE_ROOT)
if corrupt_images:
    preview = ", ".join(str(path) for path in corrupt_images[:5])
    raise DataUnavailableError(f"Found {len(corrupt_images)} corrupt image(s): {preview}")
coco = load_coco(COCO_JSON)
class_names = [category["name"] for category in coco["categories"]]
print(f"Validated {summary['images']} images, {summary['annotations']} annotations, and {summary['categories']} classes.")


In [ ]:
conversion = convert_coco_to_yolo(COCO_JSON, IMAGE_ROOT, OUTPUT_DIR)
split = split_dataset(OUTPUT_DIR, val_fraction=0.20, seed=SEED)
dataset_yaml = write_dataset_yaml(OUTPUT_DIR, class_names)
print(f"Converted {conversion['images']} images and {conversion['annotations']} annotations.")
print(f"Deterministic split: {split['train']} train / {split['val']} validation images (seed={SEED}).")
print(f"Wrote {dataset_yaml}")


In [ ]:
model = None
metrics = None
if not RUN_TRAINING:
    print("Training skipped. Set PEST_RUN_TRAINING=1 and install requirements-ml.txt to opt in.")
else:
    try:
        from ultralytics import YOLO
    except ImportError as exc:
        raise DataUnavailableError("ultralytics is required for training; install requirements-ml.txt") from exc
    training_seed(SEED)
    model = YOLO(os.environ.get("PEST_BASE_MODEL", "yolov8s.pt"))
    model.train(
        data=str(dataset_yaml),
        epochs=int(os.environ.get("PEST_EPOCHS", "170")),
        imgsz=640,
        batch=int(os.environ.get("PEST_BATCH", "16")),
        seed=SEED,
        deterministic=True,
        project=str(OUTPUT_DIR / "runs"),
        name="yolov8_pest",
    )
    model_path = OUTPUT_DIR / "yolov8_trained_model.pt"
    model.save(str(model_path))
    print(f"Saved trained model to {model_path}")


In [ ]:
if model is None:
    print("Evaluation skipped because no model was trained in this session.")
else:
    results = model.val(data=str(dataset_yaml), split="val")
    metrics = results.results_dict
    for key in ("metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"):
        if key in metrics:
            print(f"{key}: {metrics[key]:.4f}")
    print("These are validation metrics only; report a separate test set before claiming generalization.")
